In [ ]:
from pathlib import Path
from typing import Dict, Hashable, Mapping
import numpy as np
import pandas as pd

# Paths
BASE = Path('..').resolve()
ACTUAL = BASE / 'actual'
OUTPUT = BASE / 'pure_luck_goals_based'
OUTPUT.mkdir(parents=True, exist_ok=True)

LEAGUE_FILES = {
    'bundesliga': ACTUAL / 'bundesliga_actual.csv',
    'la_liga': ACTUAL / 'la_liga_actual.csv',
    'premier_league': ACTUAL / 'premier_league_actual.csv',
    'serie_a': ACTUAL / 'serie_a_actual.csv',
}

# Generate 10 random seeds for reproducibility
SEED_SEQUENCE = np.random.SeedSequence(12345)  # Master seed for reproducibility
RANDOM_SEEDS = SEED_SEQUENCE.generate_state(10)  # Generate 10 random seeds
print(f"Generated 10 random seeds: {RANDOM_SEEDS}")

In [ ]:
def empirical_goal_dists(
    df: pd.DataFrame,
    season_col: str = 'season',
    hg_col: str = 'hometeamgoals',
    ag_col: str = 'awayteamgoals'
) -> Dict[Hashable, Dict[str, np.ndarray]]:
    """Build per-season empirical goal distributions."""
    dists = {}
    tmp = df[[season_col, hg_col, ag_col]].copy()
    tmp[hg_col] = tmp[hg_col].astype(float).round().astype(int)
    tmp[ag_col] = tmp[ag_col].astype(float).round().astype(int)

    for season, grp in tmp.groupby(season_col):
        hg_counts = grp[hg_col].value_counts().sort_index()
        ag_counts = grp[ag_col].value_counts().sort_index()
        max_g = int(max(hg_counts.index.max(), ag_counts.index.max()))
        goals = np.arange(0, max_g + 1)
        hp = hg_counts.reindex(goals, fill_value=0).to_numpy(dtype=float)
        ap = ag_counts.reindex(goals, fill_value=0).to_numpy(dtype=float)
        hp_sum = hp.sum()
        ap_sum = ap.sum()
        if hp_sum == 0:
            hp = np.ones_like(goals, dtype=float)
            hp_sum = hp.sum()
        if ap_sum == 0:
            ap = np.ones_like(goals, dtype=float)
            ap_sum = ap.sum()
        dists[season] = {
            'goals': goals,
            'home_p': hp / hp_sum,
            'away_p': ap / ap_sum,
        }
    return dists


def simulate_matches_from_empirical(
    df: pd.DataFrame,
    dists: Mapping[Hashable, Dict[str, np.ndarray]],
    season_col: str = 'season',
    seed: int | None = None
) -> pd.DataFrame:
    """Sample match goals from the empirical distributions and recompute points."""
    rng = np.random.default_rng(seed)
    out = df.copy()

    sim_home = []
    sim_away = []
    for season in out[season_col]:
        dist = dists[season]
        goals = dist['goals']
        sim_home.append(rng.choice(goals, p=dist['home_p']))
        sim_away.append(rng.choice(goals, p=dist['away_p']))

    out['hometeamgoals'] = np.asarray(sim_home, dtype=int)
    out['awayteamgoals'] = np.asarray(sim_away, dtype=int)
    diff = out['hometeamgoals'] - out['awayteamgoals']
    out['hometeamresult'] = np.sign(diff).astype(int)
    out['home_team_points'] = np.where(diff > 0, 3, np.where(diff == 0, 1, 0)).astype(float)
    out['away_team_points'] = np.where(diff < 0, 3, np.where(diff == 0, 1, 0)).astype(float)
    return out


def season_rankings_from_simulation(sim_df: pd.DataFrame, seed: int) -> pd.DataFrame:
    """Collapse a simulated season into team ranks plus the random seed."""
    home = sim_df[['season', 'home_team', 'hometeamgoals', 'awayteamgoals', 'home_team_points']].copy()
    home.columns = ['season', 'team', 'goals_for', 'goals_against', 'points']
    away = sim_df[['season', 'away_team', 'awayteamgoals', 'hometeamgoals', 'away_team_points']].copy()
    away.columns = ['season', 'team', 'goals_for', 'goals_against', 'points']

    table = pd.concat([home, away], ignore_index=True)
    table = table.groupby(['season', 'team'], as_index=False).agg(
        points=('points', 'sum'),
        goals_for=('goals_for', 'sum'),
        goals_against=('goals_against', 'sum'),
    )
    table['goal_diff'] = table['goals_for'] - table['goals_against']
    table = table.sort_values(
        ['season', 'points', 'goal_diff', 'goals_for', 'team'],
        ascending=[True, False, False, False, True],
    ).reset_index(drop=True)
    table['rank'] = table.groupby('season').cumcount() + 1
    table['simulation_seed'] = int(seed_label)
    return table[['season', 'team', 'rank', 'simulation_seed']]


def simulate_10_runs_rankings(
    in_path: Path,
    seeds: list,
    out_path: Path
) -> Path:
    """Run 10 simulations and output condensed rankings with seed identifiers."""
    df = pd.read_csv(in_path)
    dists = empirical_goal_dists(df, 'season', 'hometeamgoals', 'awayteamgoals')
    rankings = []

    for i, seed in enumerate(seeds, start=1):
        print(f"  Running simulation {i}/10 with seed {seed}...")
        sim = simulate_matches_from_empirical(df, dists, 'season', seed=int(seed))
        rankings.append(season_rankings_from_simulation(sim, i))

    out_df = pd.concat(rankings, ignore_index=True)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_path, index=False)
    print(f"  Written to {out_path}")
    return out_path

In [ ]:
# Run 10 simulations for each league
written = []
for league, in_fp in LEAGUE_FILES.items():
    print(f"
Processing {league}...")
    out_fp = OUTPUT / f'{league}_goals_simulated_10runs.csv'
    path = simulate_10_runs_rankings(in_fp, RANDOM_SEEDS, out_fp)
    written.append(str(path))

print("
" + "="*60)
print("All simulations complete!")
print("="*60)
for w in written:
    print(f"  {w}")

written